In [1]:
import numpy as np
from scipy.stats import chi2, chi2_contingency, fisher_exact
from scipy.stats import chisquare

## 📌 Question 1 : Chi-Square Goodness-of-Fit (Fair Die)

A die is rolled 120 times. Observed counts per face:

| Face | 1 | 2 | 3 | 4 | 5 | 6 |
|------|---|---|---|---|---|---|
| Observed | 18 | 22 | 17 | 25 | 20 | 18 |

Test at α = 0.05 whether the die is fair using chi-square goodness-of-fit. Compute χ², df, and the critical value.

*If the die is fair every face has equal probability 1/6, so under H₀ each face should appear n/6 = 20 times. The chi-square statistic measures how far the observed counts stray from those equal expected counts. A large χ² means the die is likely biased; a small one means the deviations are just chance. Since we are comparing one set of observed frequencies against a fully specified theoretical distribution (no parameters estimated from the data), df = k - 1 = 5.*

#### Step 1 : Hypotheses

$$H_0: p_1 = p_2 = p_3 = p_4 = p_5 = p_6 = \frac{1}{6} \quad \text{(die is fair)}$$
$$H_1: \text{at least one } p_i \neq \frac{1}{6} \quad \text{(die is not fair)}$$

#### Step 2 : Why This Test

We have one categorical variable (die face) with $k = 6$ categories and a fixed theoretical distribution under $H_0$. The chi-square goodness-of-fit test is designed exactly for this: it compares a single sample of observed frequencies against expected frequencies derived from the null distribution. All expected counts = 20 ≥ 5, so the large-sample approximation is valid.

#### Step 3 : Test Statistic

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}, \quad df = k - 1$$

where $O_i$ are observed counts and $E_i = n/k$ are expected counts under a fair die.

In [2]:
observed = np.array([18, 22, 17, 25, 20, 18])  # observed face counts
n = observed.sum()  # total rolls = 120
alpha = 0.05
k = len(observed)  # number of categories

In [3]:
expected = np.array([n/k] * k)  # expected counts under fair die
chi_stat = sum((observed - expected)**2 / expected)

print('Expected counts :', expected)
print('chi_stat        :', round(chi_stat, 4))

Expected counts : [20. 20. 20. 20. 20. 20.]
chi_stat        : 2.3


#### Step 4 : Degrees of Freedom

$$df = k - 1 = 6 - 1 = 5$$

No parameters are estimated from the data (the fair-die probabilities are fully specified), so we lose only 1 degree of freedom for the total constraint $\sum O_i = n$.

In [4]:
df = k - 1  # degrees of freedom = 5
print('df :', df)

df : 5


#### Step 5 : Critical Value Method

In [5]:
chi_crit = chi2.ppf(1-alpha, df)  # right-tail critical value
print('chi_stat :', round(chi_stat, 4))
print('chi_crit :', round(chi_crit, 4))

chi_stat : 2.3
chi_crit : 11.0705


**chi_stat (2.30) < chi_crit (11.071) → Fail to Reject H₀**

> Note: chi-square goodness-of-fit is always a **right-tailed** test - only an unusually *large* χ² is evidence against H₀. The critical value uses `chi2.ppf(1 - alpha, df)`, not `1 - alpha/2`.

#### Step 6 : p-value Method

In [6]:
pval = chi2.sf(chi_stat, df)  # right-tail p-value
print('p-val :', round(pval, 4))

p-val : 0.8063


**p-val (0.806) > 0.05 → Fail to Reject H₀**

#### Final Conclusion

| Metric | Value |
|---|---|
| Total rolls $n$ | 120 |
| Categories $k$ | 6 |
| Expected per face | 20 |
| $\chi^2$ statistic | 2.30 |
| Degrees of freedom | 5 |
| $\chi^2$ critical (right-tail, α = 0.05) | 11.071 |
| p-value | 0.806 |
| Decision | **Fail to Reject H₀** |

There is insufficient evidence to conclude the die is unfair. The observed counts (18, 22, 17, 25, 20, 18) are consistent with random variation expected from a fair die (χ² = 2.30, p = 0.806).

## 📌 Question 2 : Chi-Square Test of Independence (Gender × Coffee Preference)

A survey of 200 people records gender and coffee preference (Yes / No):

|  | Coffee: Yes | Coffee: No | Row Total |
|---|---|---|---|
| Male | 60 | 40 | 100 |
| Female | 55 | 45 | 100 |
| **Col Total** | **115** | **85** | **200** |

Test independence at α = 0.05. Compute the expected frequency for each cell and verify that all expected counts are ≥ 5.

*If gender and coffee preference are independent, the proportion of coffee drinkers should be the same for males and females. Under H₀ we estimate each cell's expected count from the marginal totals alone: E = (row total × col total) / n. A large χ² signals that the observed cell counts deviate more than chance would produce if the two variables were truly unrelated.*

#### Step 1 : Hypotheses

$$H_0: \text{Gender and coffee preference are independent}$$
$$H_1: \text{Gender and coffee preference are dependent (associated)}$$

#### Step 2 : Why This Test

We have two categorical variables (gender: 2 levels; coffee preference: 2 levels) measured on a single sample of $n = 200$ individuals. The chi-square test of independence is the appropriate test: it checks whether the joint distribution of the two variables equals the product of their marginal distributions. With a 2 × 2 contingency table, $df = (r-1)(c-1) = 1$. All expected counts must be ≥ 5 for the large-sample approximation to hold; we verify this in Step 3.

#### Step 3 : Test Statistic

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}, \qquad E_{ij} = \frac{(\text{row}_i \text{ total}) \times (\text{col}_j \text{ total})}{n}$$

In [7]:
alpha = 0.05

O = np.array([[60, 40],   # [Male-Yes,   Male-No]
              [55, 45]])  # [Female-Yes, Female-No]

In [8]:
row_total = O.sum(axis=1)
col_total = O.sum(axis=0)
n = O.sum()

# E_ij = (row_i * col_j) / n
E = np.outer(row_total, col_total) / n

print('Row totals :', row_total)
print('Col totals :', col_total)
print('n          :', n)
print()
print('Expected counts E:')
print(E)
print()
print('All expected counts >= 5:', (E >= 5).all())

Row totals : [100 100]
Col totals : [115  85]
n          : 200

Expected counts E:
[[57.5 42.5]
 [57.5 42.5]]

All expected counts >= 5: True


In [9]:
chi_stat = ((O - E)**2 / E).sum()
print('chi_stat :', round(chi_stat, 4))

chi_stat : 0.5115


#### Step 4 : Degrees of Freedom

$$df = (r - 1)(c - 1) = (2 - 1)(2 - 1) = 1$$

For an $r \times c$ contingency table we lose one degree of freedom per row and per column after fixing the marginal totals, leaving $(r-1)(c-1)$ free cells.

In [10]:
r, c = O.shape
df = (r-1) * (c-1)  # degrees of freedom = 1
print('df :', df)

df : 1


#### Step 5 : Critical Value Method

In [11]:
chi_crit = chi2.ppf(1-alpha, df)  # right-tail critical value
print('chi_stat :', round(chi_stat, 4))
print('chi_crit :', round(chi_crit, 4))

chi_stat : 0.5115
chi_crit : 3.8415


**chi_stat (0.511) < chi_crit (3.841) -- Fail to Reject H₀**

> Note: the chi-square test of independence is always **right-tailed**. Only an unusually large χ² constitutes evidence of association. The critical value uses `chi2.ppf(1 - alpha, df)`.

#### Step 6 : p-value Method

In [12]:
pval = chi2.sf(chi_stat, df)  # right-tail p-value
print('p-val :', round(pval, 4))

p-val : 0.4745


**p-val (0.474) > 0.05 -- Fail to Reject H₀**

#### Final Conclusion

**Expected counts (all ≥ 5 -- large-sample approximation is valid):**

|  | Coffee: Yes | Coffee: No |
|---|---|---|
| Male | 57.5 | 42.5 |
| Female | 57.5 | 42.5 |

**Summary:**

| Metric | Value |
|---|---|
| Sample size $n$ | 200 |
| Table dimensions | 2 × 2 |
| $\chi^2$ statistic | 0.511 |
| Degrees of freedom | 1 |
| $\chi^2$ critical (right-tail, α = 0.05) | 3.841 |
| p-value | 0.474 |
| Decision | **Fail to Reject H₀** |

There is insufficient evidence to conclude that gender and coffee preference are associated. The small χ² (0.511) and large p-value (0.474) indicate that the observed cell counts are well within the range of random variation expected if the two variables were independent (χ² = 0.511, p = 0.474).

## 📌 Question 3 : Fisher's Exact Test vs Chi-Square (Clinical Trial)

A clinical trial with small cell counts records outcomes for a new drug vs placebo:

|  | Cured | Not Cured | Row Total |
|---|---|---|---|
| New Drug | 8 | 2 | 10 |
| Placebo | 4 | 6 | 10 |
| **Col Total** | **12** | **8** | **20** |

**(a)** Run chi-square without and with Yates' continuity correction.  
**(b)** Run Fisher's exact test.  
**(c)** Compare all three results.  
**(d)** State the rule: when must you use Fisher's exact test?

*The key issue here is small expected cell counts. When any expected frequency falls below 5, the chi-square large-sample approximation breaks down and Fisher's exact test -- which enumerates all possible tables with the same marginals -- is the correct choice. Yates' continuity correction partially compensates for the discreteness of the data but is still an approximation.*

#### Step 1 : Hypotheses

$$H_0: \text{Treatment and outcome are independent (drug has no effect)}$$
$$H_1: \text{Treatment and outcome are dependent (drug has an effect)}$$

#### Step 2 : Why This Test Matters Here

We have a 2x2 contingency table with only n = 20 observations. Before choosing a test we must check the expected cell counts under H0. If any expected count is below 5, the chi-square approximation is unreliable and Fisher's exact test is required.

#### Step 3 : Observed Table and Expected Counts

$$E_{ij} = \frac{(\text{row}_i \text{ total}) \times (\text{col}_j \text{ total})}{n}$$

In [13]:
alpha = 0.05

O = np.array([[8, 2],
              [4, 6]])

row_total = O.sum(axis=1)
col_total = O.sum(axis=0)
n = O.sum()

E = np.outer(row_total, col_total) / n

print('Row totals :', row_total)
print('Col totals :', col_total)
print('n          :', n)
print()
print('Expected counts E:')
print(E)
print()
print('Any expected count < 5 :', (E < 5).any())

Row totals : [10 10]
Col totals : [12  8]
n          : 20

Expected counts E:
[[6. 4.]
 [6. 4.]]

Any expected count < 5 : True


**Expected counts:**

|  | Cured | Not Cured |
|---|---|---|
| New Drug | 6.0 | 4.0 |
| Placebo | 6.0 | 4.0 |

**Two cells have expected count = 4.0 < 5.** The chi-square large-sample approximation is not strictly valid here. We proceed with both chi-square variants for comparison, but Fisher's exact test is the correct choice.

#### Step 4 : Degrees of Freedom

$$df = (r - 1)(c - 1) = (2 - 1)(2 - 1) = 1$$

In [14]:
df = (O.shape[0]-1) * (O.shape[1]-1)
print('df :', df)

df : 1


#### Step 5 (a) : Chi-Square Without Yates' Correction

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

In [15]:
chi_stat = ((O - E)**2 / E).sum()
chi_crit = chi2.ppf(1-alpha, df)
pval = chi2.sf(chi_stat, df)

print('chi_stat (no correction) :', round(chi_stat, 4))
print('chi_crit                 :', round(chi_crit, 4))
print('p-value                  :', round(pval, 4))

chi_stat (no correction) : 3.3333
chi_crit                 : 3.8415
p-value                  : 0.0679


**chi_stat (3.3333) < chi_crit (3.8415) -- Fail to Reject H0**  
**p-value (0.0679) > 0.05 -- Fail to Reject H0**

Without correction, the association is not significant at the 5% level, but it is borderline (p = 0.068).

#### Step 5 (b) : Chi-Square With Yates' Continuity Correction

Yates' correction subtracts 0.5 from each |O - E| before squaring, reducing the test statistic to partially compensate for the discreteness of count data:

$$\chi^2_{\text{Yates}} = \sum_{i,j} \frac{(|O_{ij} - E_{ij}| - 0.5)^2}{E_{ij}}$$

In [16]:
chi_stat_yates = ((np.abs(O - E) - 0.5)**2 / E).sum()
pval_yates = chi2.sf(chi_stat_yates, df)

print('chi_stat (Yates) :', round(chi_stat_yates, 4))
print('chi_crit         :', round(chi_crit, 4))
print('p-value (Yates)  :', round(pval_yates, 4))

chi_stat (Yates) : 1.875
chi_crit         : 3.8415
p-value (Yates)  : 0.1709


**chi_stat_yates (1.8750) < chi_crit (3.8415) -- Fail to Reject H0**  
**p-value (0.1709) > 0.05 -- Fail to Reject H0**

With Yates' correction the statistic drops considerably and the p-value rises to 0.171, making the result even less significant.

#### Step 5 (c) : Fisher's Exact Test

Fisher's exact test computes the exact probability of observing a table as extreme as (or more extreme than) the one observed, given fixed marginal totals. It makes no large-sample approximation and is valid for any sample size.

The p-value is the sum of hypergeometric probabilities for all tables at least as extreme as the observed one.

In [17]:
oddsratio, fisher_p = fisher_exact(O, alternative='two-sided')

print('Odds Ratio :', round(oddsratio, 4))
print('p-value    :', round(fisher_p, 4))

Odds Ratio : 6.0
p-value    : 0.1698


**p-value (0.1698) > 0.05 -- Fail to Reject H0**

Fisher's exact test gives p = 0.170. There is insufficient evidence to conclude the drug is effective at the 5% significance level.

The odds ratio of 6.0 means patients on the new drug were 6 times more likely to be cured than those on placebo, but with only 20 observations this effect is not statistically significant.

#### Step 6 : Comparison of All Three Results

| Method | Test Statistic | p-value | Decision |
|---|---|---|---|
| Chi-square (no correction) | 3.3333 | 0.0679 | Fail to Reject H0 |
| Chi-square (Yates correction) | 1.8750 | 0.1709 | Fail to Reject H0 |
| Fisher's Exact Test | -- | 0.1698 | Fail to Reject H0 |

All three methods lead to the same decision, but the p-values differ substantially. The uncorrected chi-square (p = 0.068) is the most liberal and comes closest to the 5% threshold. Yates' correction and Fisher's exact test agree closely (p ~ 0.170), which is expected: Fisher's exact test is the gold standard for small samples and Yates' correction is designed to approximate it.

> The uncorrected chi-square overstates significance here because the large-sample approximation is unreliable when expected counts fall below 5.

#### Step 7 (d) : When Must You Use Fisher's Exact Test?

Use Fisher's exact test when **any** expected cell count in the contingency table is less than 5.

More precisely:

| Condition | Recommended Test |
|---|---|
| All expected counts >= 10 | Chi-square (no correction) is reliable |
| Any expected count between 5 and 10 | Chi-square with Yates' correction |
| Any expected count < 5 | Fisher's exact test (required) |
| Total sample size n < 20 | Fisher's exact test (always preferred) |

In this problem, two cells have expected count = 4.0 < 5, so Fisher's exact test is the appropriate choice. The chi-square results are shown for comparison only.

#### Final Conclusion

| Metric | Value |
|---|---|
| Sample size n | 20 |
| Table dimensions | 2 x 2 |
| Minimum expected count | 4.0 (below threshold of 5) |
| Chi-square statistic (no correction) | 3.3333 |
| Chi-square p-value (no correction) | 0.0679 |
| Chi-square statistic (Yates) | 1.8750 |
| Chi-square p-value (Yates) | 0.1709 |
| Fisher's exact p-value | 0.1698 |
| Odds ratio | 6.0000 |
| Decision | **Fail to Reject H0** |

Because two expected cell counts (4.0) fall below 5, Fisher's exact test is the correct test. All three methods agree: there is insufficient evidence at alpha = 0.05 to conclude that the drug is more effective than placebo (Fisher p = 0.170). The odds ratio of 6.0 suggests a clinically meaningful trend, but the small sample size (n = 20) leaves the result statistically inconclusive.

## 📌 Question 4 : Chi-Square + Standardized Residuals (Age × Campaign)

A marketing survey of 413 people. We want to know: does campaign preference depend on age group?

|  | Campaign A | Campaign B | Campaign C | Total |
|---|---|---|---|---|
| Age 18–30 | 45 | 38 | 52 | 135 |
| Age 31–50 | 62 | 70 | 58 | 190 |
| Age 51+   | 28 | 35 | 25 | 88 |
| **Total** | **135** | **143** | **135** | **413** |

**(a)** Chi-square test of independence  
**(b)** Standardized residuals for each cell  
**(c)** Which cells are driving the result?  
**(d)** What |SR| threshold do we use at α = 0.05?

*The chi-square test will tell us whether there is an overall association between age and preference. But it gives us just one number - it cannot tell us which specific age-campaign combination is unusual. That is what standardized residuals are for. Each SR value is basically a z-score for one cell: how far is the observed count from what we expected, in standard deviation units. A cell with |SR| > 2 is the one worth paying attention to.*

#### Step 1 : Hypotheses

$$H_0: \text{Age group and campaign preference are independent}$$
$$H_1: \text{They are associated}$$

#### Step 2 : Data Setup

In [18]:
alpha = 0.05

O = np.array([
    [45, 38, 52],   # Age 18-30
    [62, 70, 58],   # Age 31-50
    [28, 35, 25]    # Age 51+
])

row_totals = O.sum(axis=1)
col_totals = O.sum(axis=0)
n = O.sum()

print('Row totals :', row_totals)
print('Col totals :', col_totals)
print('n          :', n)

Row totals : [135 190  88]
Col totals : [135 143 135]
n          : 413


#### Step 3 : Expected Counts

$$E_{ij} = \frac{\text{row total}_i \times \text{col total}_j}{n}$$

*If age and preference are independent, the expected count is just row proportion × column proportion × n. np.outer multiplies every row total with every column total in one go - same as doing it manually for each cell.*

In [19]:
E = np.outer(row_totals, col_totals) / n

print('Expected counts:')
print(E.round(2))
print()
print('All counts >= 5 :', (E >= 5).all())

Expected counts:
[[44.13 46.74 44.13]
 [62.11 65.79 62.11]
 [28.77 30.47 28.77]]

All counts >= 5 : True


#### Step 4 : Degrees of Freedom

$$df = (r-1)(c-1) = (3-1)(3-1) = 4$$

In [20]:
rows, cols = O.shape
df = (rows-1) * (cols-1)
print('df :', df)

df : 4


#### Step 5 (a) : Chi-Square Test

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

In [21]:
chi_stat = ((O - E)**2 / E).sum()
chi_crit = chi2.ppf(1-alpha, df)
pval = chi2.sf(chi_stat, df)

print('chi_stat :', round(chi_stat, 4))
print('chi_crit :', round(chi_crit, 4))
print('p-value  :', round(pval, 4))

chi_stat : 4.7851
chi_crit : 9.4877
p-value  : 0.3101


**chi_stat (4.7851) < chi_crit (9.4877) → Fail to Reject H₀**  
**p-value (0.3101) > 0.05 → Fail to Reject H₀**

#### Step 6 (b) : Standardized Residuals

$$SR_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij}}}$$

*Think of each SR as a z-score for that one cell. Positive means the observed count is higher than expected, negative means lower. The sqrt(E) in the denominator just scales things so all cells are comparable regardless of size.*

In [22]:
SR = (O - E) / np.sqrt(E)
print(SR.round(2))

[[ 0.13 -1.28  1.18]
 [-0.01  0.52 -0.52]
 [-0.14  0.82 -0.7 ]]


#### Step 7 (c) : Which Cells Stand Out?

In [23]:
# check each cell manually
age_labels  = ["Age 18-30", "Age 31-50", "Age 51+"]
camp_labels = ["Campaign A", "Campaign B", "Campaign C"]

print("Cells with |SR| > 2 :")
found = False
for i in range(3):
    for j in range(3):
        if abs(SR[i][j]) > 2:
            print(f"  {age_labels[i]} x {camp_labels[j]} : SR = {SR[i][j]:.4f}")
            found = True
if not found:
    print("  None - no cell is individually unusual")

Cells with |SR| > 2 :
  None - no cell is individually unusual


**No cells have |SR| > 2.** The largest is 1.28 (Age 18–30 × Campaign B), which is well within normal range. This makes sense - the overall test was also non-significant.

#### Step 8 (d) : Why |SR| > 2?

*SR values follow a standard normal distribution under H₀. For a 5% significance level, the cutoff is ±1.96 ≈ 2. So |SR| > 2 just means that cell is significant at the ~5% level on its own. For bigger tables (lots of cells), some people use |SR| > 3 to be stricter.*

| |SR| | What it means |
|---|---|
| < 1 | very close to expected, nothing unusual |
| 1 – 2 | mild deviation, not flagged |
| > 2 | notable at α = 0.05 |
| > 3 | strong, used as stricter threshold |

#### Final Conclusion

| Metric | Value |
|---|---|
| n | 413 |
| Table | 3 × 3 |
| All expected counts ≥ 5 | ✓ |
| χ² statistic | 4.7851 |
| df | 4 |
| χ² critical (α = 0.05) | 9.4877 |
| p-value | 0.3101 |
| Decision | **Fail to Reject H₀** |
| Cells with |SR| > 2 | None |

Age group and campaign preference are independent. No individual cell shows a meaningful deviation from what we expected under independence (χ² = 4.785, p = 0.310).

## 📌 Question 5 : Chi-Square Goodness-of-Fit (Snack Flavor Preference)

A snack company claims customer preference for 4 flavors is equally distributed.

| Flavor | Classic | Cheese | Spicy | BBQ |
|--------|---------|--------|-------|-----|
| Observed Sales | 52 | 47 | 61 | 40 |

Perform a chi-square goodness-of-fit test at α = 0.05.

**(a)** Perform the full chi-square test manually.

**(b)** Solve again using `scipy.stats.chisquare()`.

**(c)** Find which flavor(s) contributed most to the chi-square statistic.

*The company's claim translates directly into a null hypothesis: each of the 4 flavors should account for exactly 1/4 of all sales. Under H₀ the expected count for every flavor is n/4. A large χ² means the observed sales deviate more than chance alone would explain. Since we are comparing one observed distribution against a fully specified theoretical one (no parameters estimated from data), df = k - 1 = 3.*

#### Step 1 : Hypotheses

$$H_0: p_{\text{Classic}} = p_{\text{Cheese}} = p_{\text{Spicy}} = p_{\text{BBQ}} = \frac{1}{4} \quad \text{(preference is equally distributed)}$$
$$H_1: \text{at least one } p_i \neq \frac{1}{4} \quad \text{(preference is not equally distributed)}$$

#### Step 2 : Why This Test

We have one categorical variable (flavor) with $k = 4$ categories and a fully specified theoretical distribution under $H_0$ (equal probabilities). The chi-square goodness-of-fit test is designed exactly for this: it compares a single sample of observed frequencies against expected frequencies derived from the null distribution. Expected count per flavor = $200/4 = 50 \geq 5$, so the large-sample approximation is valid.

#### Step 3 : Test Statistic

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}, \quad df = k - 1$$

where $O_i$ are observed sales and $E_i = n/k$ are expected counts under equal preference.

In [24]:
O = np.array([52,47,61,40])

In [25]:
alpha =  0.05

In [26]:
n = O.sum()
n

np.int64(200)

In [27]:
E = np.array([n/4] * 4)
E

array([50., 50., 50., 50.])

In [28]:
chi_stat = sum((O-E)**2/E)
chi_stat

np.float64(4.68)

#### Step 4 : Degrees of Freedom

$$df = k - 1 = 4 - 1 = 3$$

No parameters are estimated from the data (equal probabilities are fully specified under $H_0$), so we lose only 1 degree of freedom for the constraint $\sum O_i = n$.

In [29]:
df = 3

#### Step 5 : Critical Value Method

In [30]:
chi_crit= chi2.ppf(1-alpha,df)
chi_crit

np.float64(7.814727903251179)

**chi_stat (4.68) < chi_crit (7.815) - Fail to Reject H₀**

> Note: chi-square goodness-of-fit is always a **right-tailed** test - only an unusually *large* χ² is evidence against H₀. The critical value uses `chi2.ppf(1 - alpha, df)`, not `1 - alpha/2`.

#### Step 6 : p-value Method

In [31]:
pval= chi2.sf(chi_stat,df)
pval

np.float64(0.19678574982211994)

**p-val (0.197) > 0.05 - Fail to Reject H₀**

#### Step 7 (b) : Verify with `scipy.stats.chisquare()`

`chisquare()` assumes equal expected frequencies by default when no `f_exp` is passed, so it matches our manual calculation exactly.

In [32]:
#part b
chi_stat, pval = chisquare(O)
chi_stat, pval

(np.float64(4.68), np.float64(0.19678574982211994))

#### Step 8 (c) : Which Flavor Contributed Most?

*Each term $(O_i - E_i)^2 / E_i$ is that flavor's individual contribution to χ². The flavor with the largest term is the one pulling the statistic the furthest from zero.*

In [33]:
contributors = (O-E)**2/E
contributors

array([0.08, 0.18, 2.42, 2.  ])

In [34]:
largest = np.argmax(contributors)
largest

np.int64(2)

In [35]:
categories = ["Classic", "Cheese", "Spicy", "BBQ"]
print(categories[largest])

Spicy


#### Final Conclusion

| Metric | Value |
|---|---|
| Total sales $n$ | 200 |
| Categories $k$ | 4 |
| Expected per flavor | 50 |
| $\chi^2$ statistic | 4.68 |
| Degrees of freedom | 3 |
| $\chi^2$ critical (right-tail, α = 0.05) | 7.815 |
| p-value | 0.197 |
| Decision | **Fail to Reject H₀** |
| Largest contributor | **Spicy** |

There is insufficient evidence to conclude that customer preference is unequally distributed across the four flavors (χ² = 4.68, p = 0.197). Although Spicy had the highest observed sales (61) and contributed the most to χ², the deviation is within the range of random variation expected under equal preference.